# 08_02 Classification fine-tuning: a spam filter

*📖 Book: chapter 6 (fine-tuning for classification)*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **after the main course**: the adult's short course for one job
>
> Project files this connects to:
>
> - `src/homer_ai/finetune.py`
> - `src/homer_ai/pretrained.py`
> - `src/homer_ai/gpt.py`
>
> **This notebook:** GPT-2 writes text. How do we make it sort text messages into spam and not spam, by training only a small part of it for about a minute?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- What classification is, and how it differs from writing text
- How to prepare labelled data: balance it, split it three ways, pad it to one length
- How to swap the output head for a 2-score one, and freeze most of the model
- How to measure accuracy and read a confusion table
- How much one minute of training on a laptop CPU gets you

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **classification** | sorting each input into one of a fixed set of groups (classes) | a message is `spam` or `ham` |
| **label** | the right answer that comes with a training example | `spam` |
| **ham** | the data set's word for a normal message, one that isn't spam | `Ok lar... Joking wif u oni...` |
| **balanced** | a data set with the same number of examples of each class | 747 spam and 747 ham |
| **test set** | examples kept aside until the very end, to measure the finished model once | 300 messages |
| **padding** | filler tokens added to short texts so every row of a batch has the same length | token 50256 repeated |
| **classification head** | an output layer with one score per class instead of one per token | `nn.Linear(768, 2)` |
| **frozen** | a weight that training isn't allowed to change (`requires_grad = False`) | blocks 0 to 10 of GPT-2 |
| **accuracy** | the share of examples the model gets right | 281 of 300 = 93.7% |
| **confusion table** | a small table that counts, for each true class, what the model said | "spam called ham: 12" |

## 3. The idea

### ❓ What job are we teaching, and why this one?

**Classification**: sorting each input into one of a fixed set of groups. Ours is the classic one, a spam filter. A text message goes in, and one of two answers comes out: `spam`, or `ham` (the data set's word for a normal message).

It's a good first job because the answer is short and easy to check. A written text can be good in a hundred ways. A label is right or wrong, so we can count.

In the picture of [08_00](08_00_chapter_overview.ipynb): GPT-2 is the adult who already reads English. We give them a short course, a thousand example messages with the right answer on each. And while they take it, they keep nearly everything they know and learn only the new bit. That's what **freezing** is, further down.

### ❓ A GPT writes text. How can it sort messages?

By changing one part. Everything up to the final layer norm stays. That part turns a message into 768 numbers per token, and those numbers already carry a lot about what the text says. Only the last layer changes: the output head.

```
writing:      … → final norm → output head (768 → 50,257 scores: which token comes next?)
classifying:  … → final norm → new head    (768 → 2 scores: ham or spam?)
```

The new head starts with random numbers, so at first it guesses. Training it, plus a small part of the model under it, is the whole job.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: what does the data look like?

It's the **SMS Spam Collection** by Tiago Almeida and José María Gómez Hidalgo: 5,574 real English text messages, each marked `ham` or `spam` by hand. It comes from the UCI Machine Learning Repository, under the CC BY 4.0 licence (free to use if you name the authors). `load_sms_spam` downloads the 0.2 MB zip once to `~/.cache/homer-ai/sms_spam/`, checks its SHA-256, and reads it. The helper code lives in `src/homer_ai/`: `pretrained.py`, `finetune.py` and `lora.py`, imported like the rest of the project.

👉 **Predict first:** of 5,574 messages, how many do you think are spam: about half?

In [ ]:
from collections import Counter

from homer_ai.finetune import load_sms_spam

messages = load_sms_spam()  # a list of (label, text) pairs
print(len(messages), "messages:", dict(Counter(label for label, _ in messages)))
print()
for label, text in messages[1:4]:
    print(f"{label:5}| {text[:90]}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Not half at all: 4,827 ham and only 747 spam.

```
ham  | Ok lar... Joking wif u oni...
spam | Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to…
ham  | U dun say so early hor... U c already then say...
```

The ham is chatty and full of slang (much of it from Singapore). The spam shouts about prizes and phone numbers. A person tells them apart at a glance.

</details>

### ❓ Step 2: why can't we train on it as it is?

Because 87% of it is ham. A lazy model that answers "ham" every time would be right 87% of the time and would have learned nothing. We want a data set where guessing gives 50%, so that any score above 50% means something.

So we **balance** it: keep all 747 spam, and pick 747 ham at random (with a fixed seed, so you get the same ones as I do). Then we cut the 1,494 messages into three parts:

```
training   70%   the model learns from these
validation 10%   we peek at these while we work (05_05)
test       20%   we look at these ONCE, at the very end
```

The **test set** is new. In chapter 5 we had only training and validation. But if we tune settings while watching the validation score, that score slowly stops being a fair exam. The test set is the exam nobody has seen.

In [ ]:
from homer_ai.finetune import balance, split

balanced = balance(messages, seed=123)
train, val, test = split(balanced, train=0.7, val=0.1)
for name, part in [("training", train), ("validation", val), ("test", test)]:
    counts = Counter(label for label, _ in part)
    ham, spam = counts["ham"], counts["spam"]
    print(f"{name:10} {len(part):>5} messages   ham {ham:>3}   spam {spam:>3}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

1,045 for training, 149 for validation, 300 for the test. Each part is close to half ham and half spam (not exactly, because the cut is by position in a shuffled list).

</details>

### ❓ Step 3: how do messages of different lengths go into one tensor?

A batch is a table, so every row needs the same length ([05_07](../05_training_on_the_odyssey/05_07_train_the_gpt.ipynb)). Messages don't do us that favour. So we pick one length and make every message fit: cut the long ones, and fill the short ones with a filler token. That's **padding**. As filler we use GPT-2's `<|endoftext|>`, token 50256.

Here it is for two real messages and a length of 12:

```
text  : Ok lar... Joking wif u oni...
ids   : 18690  26371  986  …  72  986              (11 tokens)
padded: 18690  26371  986  …  72  986  50256       (12: one filler added)

text  : Free entry in 2 a wkly comp to win FA Cup final tkts …
ids   : 11146  5726  287  362  257  …              (50 tokens)
padded: 11146  5726  287  362  257  …              (12: cut after the twelfth)
```

👉 **Predict first:** `Ok lar... Joking wif u oni...` has 6 words. Will it be 6 tokens?

In [ ]:
import tiktoken

from homer_ai.finetune import PAD_ID, encode_messages

encoding = tiktoken.get_encoding("gpt2")
for label, text in messages[1:3]:
    ids = encoding.encode(text)
    padded, _ = encode_messages([(label, text)], encoding, length=12)
    print("text  :", text[:60])
    print("tokens:", "|".join(encoding.decode([i]) for i in ids[:12]))
    print(f"ids   : {len(ids)} tokens, starting {ids[:4]}")
    print("padded:", padded[0].tolist())
    print()
print("the filler token:", PAD_ID, repr(encoding.decode([PAD_ID])))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Eleven tokens, not six. GPT-2's BPE cuts slang into pieces (`Ok| lar|...| J|oking| w|if| u| on|i|...`; a token often starts with its space), so the short message needed one filler, 50256, to reach 12. The spam message has 50 tokens and was cut after the twelfth.

</details>

### ❓ Which length do we pick for real?

Long enough to keep nearly every message whole, and no longer, because every extra token costs time on every step. Let's look at the lengths in the training set.

In [ ]:
lengths = sorted(len(encoding.encode(text)) for _, text in train)
middle = lengths[len(lengths) // 2]
print("shortest:", lengths[0], " middle:", middle, " longest:", lengths[-1])

LENGTH = 64
longer = sum(n > LENGTH for n in lengths)
print(f"longer than {LENGTH} tokens: {longer} of {len(lengths)} messages")

train_ids, train_labels = encode_messages(train, encoding, LENGTH)
val_ids, val_labels = encode_messages(val, encoding, LENGTH)
test_ids, test_labels = encode_messages(test, encoding, LENGTH)
print("train_ids:", tuple(train_ids.shape), " train_labels:", tuple(train_labels.shape))
print("first 10 labels:", train_labels[:10].tolist(), " (0 = ham, 1 = spam)")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The middle message has 32 tokens and the longest 87. With `LENGTH = 64` only 11 of the 1,045 messages get cut, and shorter rows make every step faster. The training set is now one table of shape `(1045, 64)`, with 1,045 labels: 0 for ham, 1 for spam.

</details>

### ❓ Step 4: how do we swap the head?

We load GPT-2 into our class ([08_01](08_01_loading_gpt2.ipynb), now one call), and replace `model.output_head` with a new `nn.Linear(768, 2)`.

The model still gives scores for **every** token of the message. Which ones do we read? The **last** token's. Because of the causal mask ([03_03](../03_attention/03_03_hiding_the_future.ipynb)), a token only sees what comes before it, so the last one is the only one that has seen the whole message.

```
ids     (1, 64)        one message, 64 tokens
model   (1, 64, 2)     2 scores at every token
last    (1, 2)         the 2 scores at the last token:  [ham score, spam score]
```

In [ ]:
import tiktoken
import torch

from homer_ai.pretrained import load_gpt2

encoding = tiktoken.get_encoding("gpt2")
model = load_gpt2()  # GPT-2 small inside our own GPTModel (08_01); read from the cache

In [ ]:
from torch import nn

print("head before:", model.output_head)
torch.manual_seed(123)
model.output_head = nn.Linear(768, 2)  # 2 scores: ham, spam. Random for now.
print("head after: ", model.output_head)

one = train_ids[:1]  # one message, shape (1, 64)
with torch.no_grad():
    scores = model(one)
print()
print("scores at every token:", tuple(scores.shape))
print("scores at the last one:", scores[:, -1, :])

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The head went from `768 → 50257` to `768 → 2`, and the output from one score per vocabulary token to two scores per position: `(1, 64, 2)`. The last position's two numbers are the model's answer. Right now they mean nothing, because the head is random.

</details>

### ❓ Step 5: what do we freeze, and why?

**Freezing** a weight means telling PyTorch not to change it: `requires_grad = False`. No nudge is worked out for it, and the optimizer leaves it alone.

We freeze almost everything. The adult keeps what they know. We train only three parts at the top:

```
token + position tables      frozen
blocks 0 … 10                frozen     ← everything GPT-2 knows about English
block 11 (the last one)      TRAINED
final layer norm             TRAINED
new head (768 → 2)           TRAINED    ← starts random
```

Why not train only the head? You can, and it works a little. Raschka's book reports clearly better results when the last block may move too, because then it can reshape its 768 numbers for this one job. And why not train everything? It's slower, and with only 1,045 examples a fully free model can forget what it knew.

There's a speed bonus too. Working out nudges runs backwards from the loss, and it can stop at the first frozen block. So we pay for the backward pass of one block, not twelve.

👉 **Predict first:** one block of twelve, plus two small parts. What share of all the weights will be trained?

In [ ]:
for parameter in model.parameters():
    parameter.requires_grad = False  # freeze everything...
for part in (model.blocks[-1], model.final_norm, model.output_head):
    for parameter in part.parameters():
        parameter.requires_grad = True  # ...then unfreeze three parts

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trained: {trainable:>11,}")
print(f"all:     {total:>11,}")
print(f"share:   {trainable / total:.1%}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

7,090,946 of 124,441,346, so 5.7%. (The total is smaller than the 163 million of 08_01, because the big 50,257-row head is gone and a 2-row one took its place.)

`make_classifier` in `finetune.py` is steps 4 and 5 in one call.

</details>

### ❓ Step 6: how good is it before any training?

We need a number to beat. **Accuracy** is the share of messages the model gets right. The model's answer is whichever of its two scores is higher (`argmax`).

👉 **Predict first:** the head is random and the data is balanced. What accuracy do you expect?

In [ ]:
from homer_ai.finetune import accuracy

before = accuracy(model, val_ids, val_labels)
print(f"validation accuracy, untrained head: {before:.1%}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

54.4%, which is coin-flipping on 149 messages. A random head knows nothing, as it should.

</details>

### ❓ Step 7: what does the training loop look like?

It's the loop of chapter 5: guess, measure the loss, work out the nudges, nudge. Only the first two lines changed.

```python
scores = model(batch_ids)[:, -1, :]            # (8, 2): the last token's 2 scores
loss = F.cross_entropy(scores, batch_labels)   # the right answer is 0 or 1
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

The loss is still cross-entropy ([05_01](../05_training_on_the_odyssey/05_01_loss_and_epochs.ipynb)). Before, it asked "how much chance did you give the right next token, out of 50,257?". Now it asks "how much chance did you give the right label, out of 2?". Pure guessing gives `ln(2) = 0.693`.

`train_classifier` wraps that loop: batches of 8 messages, shuffled each epoch, AdamW with a learning rate of 0.00005 (small, because we're adjusting a trained model, not building one). We run 3 epochs, which is 390 steps. It takes about 65 seconds on my laptop.

👉 **Predict first:** the loss starts near 0.693 or above. Where will it be after 3 epochs?

In [ ]:
import time

from homer_ai.finetune import train_classifier

started = time.time()
history = train_classifier(
    model, train_ids, train_labels, epochs=3, lr=5e-5, batch_size=8, report_every=30
)
train_seconds = time.time() - started
print(f"training took {train_seconds:.0f} seconds")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The loss starts above 1 (a random head can be confidently wrong, which costs more than guessing), drops under 0.693 within the first epoch, and ends near 0.15. On my laptop: about 65 seconds.

</details>

### ❓ Step 8: did it learn? The three accuracies

Now the exam. Training accuracy shows whether it learned the examples. Validation accuracy shows whether that carries over to new messages. And the test accuracy is the one we report, because we never looked at those 300 messages while working.

In [ ]:
results = {
    "training (first 300)": accuracy(model, train_ids[:300], train_labels[:300]),
    "validation": accuracy(model, val_ids, val_labels),
    "test": accuracy(model, test_ids, test_labels),
}
for name, value in results.items():
    print(f"{name:22} {value:.1%}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

On my laptop: training 97.0%, validation 94.6%, test **93.7%**. From a coin flip to 281 of 300 right, in about a minute, with 94% of the model frozen.

Training is a bit higher than the other two. That's the gap of [05_03](../05_training_on_the_odyssey/05_03_memorising_vs_understanding.ipynb): it knows its own examples a little better than new ones. It's small, so the model mostly learned the job and not the examples.

For scale: Raschka's book trains 5 epochs on messages padded to 120 tokens and reports about 95.7% on its test set. We train less and cut at 64 tokens, and land two points lower.

</details>

### ❓ Which mistakes does it make?

One number hides what kind of mistakes they are, and for a spam filter that matters. A real message thrown into the spam folder is worse than one spam getting through.

A **confusion table** counts both kinds. Each row is what the message really was, each column what the model said.

In [ ]:
from homer_ai.finetune import class_scores

with torch.no_grad():
    batches = [test_ids[i : i + 50] for i in range(0, 300, 50)]
    guesses = torch.cat([class_scores(model, b).argmax(dim=-1) for b in batches])

print(f"{'':14}{'said ham':>10}{'said spam':>11}")
for truth, name in enumerate(["really ham", "really spam"]):
    row = [((test_labels == truth) & (guesses == said)).sum().item() for said in (0, 1)]
    print(f"{name:14}{row[0]:>10}{row[1]:>11}")

print()
wrong = (guesses != test_labels).nonzero().flatten().tolist()
for i in wrong[:4]:
    label, text = test[i]
    print(f"really {label:4} | {text[:80]}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

On my laptop the 19 mistakes split like this:

```
                said ham  said spam
really ham           142          7
really spam           12        139
```

It called a real message spam 7 times, and let spam through 12 times. Look at the wrong ones it printed. They're the odd cases: spam written in a chatty voice (`Did you hear about the new "Divorce Barbie"?`), and a real message that sounds like a chain letter (`Send this to ur friends and receive something…`). I'd have hesitated on some of those myself.

</details>

### ❓ Can I try my own messages?

Yes. Encode the text the same way (same length, same padding), and read the last token's scores. `softmax` turns the two scores into chances that add up to 1.

In [ ]:
my_messages = [
    "You are a winner! You have been specially selected to receive $1000 cash. Call!",
    "Hey, are we still on for dinner tonight? Let me know.",
    "Tell me, O Muse, of that ingenious hero who travelled far and wide.",
]
ids, _ = encode_messages([("ham", text) for text in my_messages], encoding, LENGTH)
with torch.no_grad():
    chances = torch.softmax(class_scores(model, ids), dim=-1)
for text, (_ham, spam) in zip(my_messages, chances.tolist(), strict=True):
    print(f"spam {spam:5.1%} | {text[:70]}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The prize message gets 64.9% spam and the dinner message 0.2%. The line from Butler's Odyssey gets 0.4%, so ham. Keep in mind what that means, though. The model never saw anything like Homer in training, and a classifier always answers, even about text far from its training data. Here the answer happens to be sensible.

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Why read the last token and not the average of all of them?

Both can work. The last token is the simple choice for a GPT, because it's the only one that saw the whole text. In our padded rows the last token is usually a filler, and that's fine: the filler sits after the message, so it has seen all of it, and training teaches the last block to gather the answer there.

### ❓ Why is the learning rate so small?

0.00005, against 0.001 when we trained the Homer GPT from scratch. GPT-2's weights are already good. Big steps would knock them about and lose what they hold. Small steps adjust them.

### ❓ Could I get more than 93.7%?

Probably, by training longer. Change `epochs=3` to `epochs=5`, or set `LENGTH` to 87 so no message is cut, and run the notebook again. Each makes it slower. Judge by the validation accuracy while you try things, and look at the test accuracy once at the end.

### ❓ Is 93.7% good for a spam filter?

For a minute of training, yes. For a product, no: on this data set, much simpler methods that only count words are reported at about 97%. Fine-tuning a language model pays off on jobs where the meaning of a sentence matters more than which words are in it.

### ❓ Where is the trained classifier saved?

Nowhere. It lives in this notebook's memory, and training it again takes a minute. [08_04](08_04_lora.ipynb) saves a fine-tuned model, to the cache folder outside the project, and shows how small such a file can be.

## 5. Recap

- Classification sorts an input into one of a few classes. A GPT does it with a new output head (768 → 2), read at the last token.
- Labelled data needs care first: balance the classes, split into training, validation and test, pad to one length.
- Freezing locks most weights. We trained the last block, the final norm and the head: 5.7% of the model.
- Three epochs (about a minute on a laptop CPU) took accuracy from a coin flip to 93.7% on unseen messages.
- A confusion table shows which kind of mistake a classifier makes.

**Next:** [08_03: Instruction fine-tuning](08_03_instructions.ipynb).